In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 3 : Distributed Machine Learning Models
# Setup
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import time

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task3")
    .master("local[*]")
    .config("spark.driver.memory","10g")
    .config("spark.sql.shuffle.partitions","200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version : {spark.version}")

# ──────────────────────────────────────────────────────────────────────────────
# Required Libraries
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml import PipelineModel

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

from pyspark.ml.tuning import (
    CrossValidator,
    ParamGridBuilder
)

from pyspark.storagelevel import StorageLevel

# ──────────────────────────────────────────────────────────────────────────────
# Project Directories
# ──────────────────────────────────────────────────────────────────────────────

BASE = "/content/drive/MyDrive/7006SCN"

PROCESSED = f"{BASE}/data/processed"

MODELS = f"{BASE}/models"

OUTPUT = f"{BASE}/outputs"

print("Project directories configured ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark Version : 4.0.3
Project directories configured ✓


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# Load HMDA Clean Dataset
# ──────────────────────────────────────────────────────────────────────────────

df_clean = spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print("="*80)
print("HMDA CLEAN DATASET LOADED")
print("="*80)

print(f"Rows    : {df_clean.count():,}")
print(f"Columns : {len(df_clean.columns)}")

HMDA CLEAN DATASET LOADED
Rows    : 11,564,178
Columns : 123


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Load Spark Preprocessing Pipeline
# ──────────────────────────────────────────────────────────────────────────────

pipeline_model = PipelineModel.load(
    f"{MODELS}/preprocessing_pipeline"
)

print("Preprocessing Pipeline Loaded ✓")
# ──────────────────────────────────────────────────────────────────────────────
# Generate Feature Vector
# ──────────────────────────────────────────────────────────────────────────────

df_model = pipeline_model.transform(df_clean)

print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)

df_model.select(
    "loan_approval",
    "features"
).show(5, truncate=False)
# ──────────────────────────────────────────────────────────────────────────────
# Train-Test Split
# ──────────────────────────────────────────────────────────────────────────────

train, test = df_model.randomSplit(
    [0.8,0.2],
    seed=7006
)

# -------------------------------------------------------------------------
# Representative Sampling (10%)
# -------------------------------------------------------------------------

train = (
    train.sample(
        withReplacement=False,
        fraction=0.10,
        seed=7006
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test = (
    test.sample(
        withReplacement=False,
        fraction=0.10,
        seed=7006
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print()

print("="*80)
print("Sampled Dataset")
print("="*80)

print(f"Training : {train.count():,}")

print(f"Testing  : {test.count():,}")


# ──────────────────────────────────────────────────────────────────────────────
# Evaluation Metrics
# ──────────────────────────────────────────────────────────────────────────────

ev_acc = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="accuracy"
)

ev_f1 = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="f1"
)

ev_auc = BinaryClassificationEvaluator(
    labelCol="loan_approval",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

results = []

print("Evaluators Ready ✓")
# ──────────────────────────────────────────────────────────────────────────────
# Commit 1 (LR): Baseline Logistic Regression
# feat(task3/lr): baseline Logistic Regression model
#
# Justification:
# Logistic Regression is selected as a strong interpretable baseline classifier
# for predicting mortgage loan approval. The model estimates approval
# probabilities from applicant, loan and property characteristics and serves
# as a benchmark before evaluating more complex tree-based ensemble models.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml.classification import LogisticRegression

print("=" * 80)
print("COMMIT 1 : BASELINE LOGISTIC REGRESSION")
print("=" * 80)

# -----------------------------------------------------------------------------
# Create Logistic Regression Model
# -----------------------------------------------------------------------------

lr = LogisticRegression(
    featuresCol="features",
    labelCol="loan_approval",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True
)

print("Training Logistic Regression Model...")

start = time.time()

lr_model = lr.fit(train)

training_time = time.time() - start

print(f"Training completed in {training_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Generate Predictions
# -----------------------------------------------------------------------------

pred_lr = lr_model.transform(test)

# -----------------------------------------------------------------------------
# Evaluate Model
# -----------------------------------------------------------------------------

accuracy = ev_acc.evaluate(pred_lr)
f1_score = ev_f1.evaluate(pred_lr)
roc_auc = ev_auc.evaluate(pred_lr)

print("\n")
print("=" * 80)
print("BASELINE LOGISTIC REGRESSION RESULTS")
print("=" * 80)

print(f"Accuracy      : {accuracy:.4f}")
print(f"F1-Score      : {f1_score:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"Training Time : {training_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Save Baseline Model
# -----------------------------------------------------------------------------

lr_model.write() \
    .overwrite() \
    .save(f"{MODELS}/lr_baseline")

print("Baseline Logistic Regression model saved ✓")

# -----------------------------------------------------------------------------
# Store Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Logistic Regression",
    "Version": "Baseline",
    "Accuracy": round(accuracy, 4),
    "F1-Score": round(f1_score, 4),
    "ROC-AUC": round(roc_auc, 4),
    "Training Time (s)": round(training_time, 2)
})

print("=" * 80)
print("COMMIT 1 COMPLETED")
print("=" * 80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 2 (LR): Hyperparameter Tuning using CrossValidator
# feat(task3/lr): hyperparameter optimisation using CrossValidator
#
# Justification:
# Hyperparameter tuning is performed using Spark ML CrossValidator to identify
# the optimal regularisation parameters for Logistic Regression while improving
# the model's generalisation performance.
# ──────────────────────────────────────────────────────────────────────────────

print("="*80)
print("COMMIT 2 : LOGISTIC REGRESSION HYPERPARAMETER TUNING")
print("="*80)

# -----------------------------------------------------------------------------
# Hyperparameter Grid
# -----------------------------------------------------------------------------

paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.10])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

print(f"Parameter combinations : {len(paramGrid)}")

# -----------------------------------------------------------------------------
# CrossValidator
# -----------------------------------------------------------------------------

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=ev_auc,
    numFolds=3,
    parallelism=2,
    seed=7006
)

print("\nStarting Cross Validation...")

start = time.time()

cv_model = cv.fit(train)

cv_time = time.time() - start

print(f"\nCross Validation completed in {cv_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Best Model
# -----------------------------------------------------------------------------

best_lr = cv_model.bestModel

print("\nBest Hyperparameters")
print("-"*40)

print(f"regParam        : {best_lr._java_obj.getRegParam()}")

print(f"elasticNetParam : {best_lr._java_obj.getElasticNetParam()}")

# -----------------------------------------------------------------------------
# Evaluate Best Model
# -----------------------------------------------------------------------------

pred_cv = best_lr.transform(test)

cv_acc = ev_acc.evaluate(pred_cv)

cv_f1 = ev_f1.evaluate(pred_cv)

cv_auc = ev_auc.evaluate(pred_cv)

print("\n")
print("="*80)
print("TUNED LOGISTIC REGRESSION RESULTS")
print("="*80)

print(f"Accuracy      : {cv_acc:.4f}")
print(f"F1-Score      : {cv_f1:.4f}")
print(f"ROC-AUC       : {cv_auc:.4f}")
print(f"Training Time : {cv_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Save Best Model
# -----------------------------------------------------------------------------

best_lr.write() \
    .overwrite() \
    .save(f"{MODELS}/lr_cv_best")

print("\nBest Logistic Regression model saved ✓")

# -----------------------------------------------------------------------------
# Update Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Logistic Regression",
    "Version": "CrossValidator",
    "Accuracy": round(cv_acc,4),
    "F1-Score": round(cv_f1,4),
    "ROC-AUC": round(cv_auc,4),
    "Training Time (s)": round(cv_time,2),
    "Best Params":
        f"regParam={best_lr._java_obj.getRegParam()}, "
        f"elasticNet={best_lr._java_obj.getElasticNetParam()}"
})

print("="*80)
print("COMMIT 2 COMPLETED")
print("="*80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 3 (DT): Decision Tree Classification
# feat(task3/dt): implement and optimise Decision Tree classifier
#
# Justification:
# Decision Tree captures non-linear relationships and decision boundaries
# within mortgage approval data. Hyperparameter tuning using CrossValidator
# identifies the optimal tree depth and minimum node size to improve
# classification performance while reducing overfitting.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml.classification import DecisionTreeClassifier

print("="*80)
print("COMMIT 3 : DECISION TREE CLASSIFIER")
print("="*80)

# -----------------------------------------------------------------------------
# Baseline Decision Tree
# -----------------------------------------------------------------------------

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="loan_approval",
    predictionCol="prediction",
    probabilityCol="probability",
    seed=7006
)

# -----------------------------------------------------------------------------
# Hyperparameter Grid
# -----------------------------------------------------------------------------

paramGrid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10])
    .addGrid(dt.minInstancesPerNode, [1, 5])
    .build()
)

print(f"Parameter combinations : {len(paramGrid)}")

# -----------------------------------------------------------------------------
# CrossValidator
# -----------------------------------------------------------------------------

cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=paramGrid,
    evaluator=ev_auc,
    numFolds=3,
    parallelism=2,
    seed=7006
)

print("\nTraining Decision Tree...")

start = time.time()

dt_cv = cv.fit(train)

dt_time = time.time() - start

print(f"Training completed in {dt_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Best Model
# -----------------------------------------------------------------------------

best_dt = dt_cv.bestModel

print("\nBest Hyperparameters")
print("-"*40)

print(f"maxDepth            : {best_dt.getMaxDepth()}")
print(f"minInstancesPerNode : {best_dt.getMinInstancesPerNode()}")

# -----------------------------------------------------------------------------
# Prediction
# -----------------------------------------------------------------------------

pred_dt = best_dt.transform(test)

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

accuracy = ev_acc.evaluate(pred_dt)
f1_score = ev_f1.evaluate(pred_dt)
roc_auc = ev_auc.evaluate(pred_dt)

print("\n" + "="*80)
print("DECISION TREE RESULTS")
print("="*80)

print(f"Accuracy      : {accuracy:.4f}")
print(f"F1-Score      : {f1_score:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"Training Time : {dt_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Save Best Model
# -----------------------------------------------------------------------------

best_dt.write() \
    .overwrite() \
    .save(f"{MODELS}/dt_best")

print("\nDecision Tree model saved ✓")

# -----------------------------------------------------------------------------
# Store Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Decision Tree",
    "Accuracy": round(accuracy, 4),
    "F1-Score": round(f1_score, 4),
    "ROC-AUC": round(roc_auc, 4),
    "Training Time (s)": round(dt_time, 2),
    "Best Hyperparameters":
        f"maxDepth={best_dt.getMaxDepth()}, "
        f"minInstances={best_dt.getMinInstancesPerNode()}"
})

print("="*80)
print("COMMIT 3 COMPLETED")
print("="*80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 4 (RF): Random Forest Classification
# feat(task3/rf): implement and optimise Random Forest classifier
#
# Justification:
# Random Forest is an ensemble learning algorithm that combines multiple
# Decision Trees using bootstrap aggregation (bagging). It improves predictive
# performance by reducing overfitting and provides robust feature importance
# estimates for mortgage loan approval prediction.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml.classification import RandomForestClassifier

print("="*80)
print("COMMIT 4 : RANDOM FOREST CLASSIFIER")
print("="*80)

# -----------------------------------------------------------------------------
# Baseline Random Forest
# -----------------------------------------------------------------------------

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="loan_approval",
    predictionCol="prediction",
    probabilityCol="probability",
    seed=7006,
    subsamplingRate=0.8
)

# -----------------------------------------------------------------------------
# Hyperparameter Grid
# -----------------------------------------------------------------------------

paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

print(f"Parameter combinations : {len(paramGrid)}")

# -----------------------------------------------------------------------------
# CrossValidator
# -----------------------------------------------------------------------------

cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=ev_auc,
    numFolds=3,
    parallelism=2,
    seed=7006
)

print("\nTraining Random Forest...")

start = time.time()

rf_cv = cv.fit(train)

rf_time = time.time() - start

print(f"\nTraining completed in {rf_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Best Model
# -----------------------------------------------------------------------------

best_rf = rf_cv.bestModel

print("\nBest Hyperparameters")
print("-"*40)

print(f"numTrees : {best_rf.getNumTrees}")
print(f"maxDepth : {best_rf.getMaxDepth()}")

# -----------------------------------------------------------------------------
# Prediction
# -----------------------------------------------------------------------------

pred_rf = best_rf.transform(test)

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

accuracy = ev_acc.evaluate(pred_rf)
f1_score = ev_f1.evaluate(pred_rf)
roc_auc = ev_auc.evaluate(pred_rf)

print("\n" + "="*80)
print("RANDOM FOREST RESULTS")
print("="*80)

print(f"Accuracy      : {accuracy:.4f}")
print(f"F1-Score      : {f1_score:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"Training Time : {rf_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Feature Importance
# -----------------------------------------------------------------------------

import pandas as pd

feature_importance = pd.DataFrame({
    "Feature_Index": list(range(len(best_rf.featureImportances))),
    "Importance": best_rf.featureImportances.toArray()
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.to_csv(
    f"{OUTPUT}/rf_feature_importance.csv",
    index=False
)

print("\nFeature importance exported ✓")

# -----------------------------------------------------------------------------
# Save Best Model
# -----------------------------------------------------------------------------

best_rf.write() \
    .overwrite() \
    .save(f"{MODELS}/rf_best")

print("Random Forest model saved ✓")

# -----------------------------------------------------------------------------
# Store Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Random Forest",
    "Accuracy": round(accuracy,4),
    "F1-Score": round(f1_score,4),
    "ROC-AUC": round(roc_auc,4),
    "Training Time (s)": round(rf_time,2),
    "Best Hyperparameters":
        f"numTrees={best_rf.getNumTrees}, "
        f"maxDepth={best_rf.getMaxDepth()}"
})

print("="*80)
print("COMMIT 4 COMPLETED")
print("="*80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 3-Commit 5 (GBT): Gradient Boosted Trees Classification
# feat(task3/gbt): implement and optimise Gradient Boosted Trees classifier
#
# Justification:
# Gradient Boosted Trees improve predictive performance by sequentially
# combining weak Decision Trees. Each tree corrects the errors of the
# previous trees, resulting in a powerful ensemble model for mortgage
# loan approval prediction.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.ml.classification import GBTClassifier

print("="*80)
print("COMMIT 5 : GRADIENT BOOSTED TREES")
print("="*80)

# -----------------------------------------------------------------------------
# Baseline GBT
# -----------------------------------------------------------------------------


gbt = GBTClassifier(
    featuresCol="features",
    labelCol="loan_approval",
    predictionCol="prediction",
    seed=7006
)

# -----------------------------------------------------------------------------
# Hyperparameter Grid
# -----------------------------------------------------------------------------

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [5, 8])
    .addGrid(gbt.maxIter, [20])
    .addGrid(gbt.stepSize, [0.05, 0.10])
    .build()
)

print(f"Parameter combinations : {len(paramGrid)}")

# -----------------------------------------------------------------------------
# CrossValidator
# -----------------------------------------------------------------------------

cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=paramGrid,
    evaluator=ev_auc,
    numFolds=3,
    parallelism=2,
    seed=7006
)

print("\nTraining Gradient Boosted Trees...")

start = time.time()

gbt_cv = cv.fit(train)

gbt_time = time.time() - start

print(f"\nTraining completed in {gbt_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Best Model
# -----------------------------------------------------------------------------

best_gbt = gbt_cv.bestModel

print("\nBest Hyperparameters")
print("-"*40)

print(f"maxDepth : {best_gbt.getMaxDepth()}")
print(f"maxIter  : {best_gbt.getMaxIter()}")
print(f"stepSize : {best_gbt.getStepSize()}")

# -----------------------------------------------------------------------------
# Prediction
# -----------------------------------------------------------------------------

pred_gbt = best_gbt.transform(test)

# -----------------------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------------------

accuracy = ev_acc.evaluate(pred_gbt)
f1_score = ev_f1.evaluate(pred_gbt)
roc_auc = ev_auc.evaluate(pred_gbt)

print("\n" + "="*80)
print("GRADIENT BOOSTED TREES RESULTS")
print("="*80)

print(f"Accuracy      : {accuracy:.4f}")
print(f"F1-Score      : {f1_score:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")
print(f"Training Time : {gbt_time:.2f} seconds")

# -----------------------------------------------------------------------------
# Save Best Model
# -----------------------------------------------------------------------------

best_gbt.write() \
    .overwrite() \
    .save(f"{MODELS}/gbt_best")

print("\nGradient Boosted Trees model saved ✓")

# -----------------------------------------------------------------------------
# Store Results
# -----------------------------------------------------------------------------

results.append({
    "Model": "Gradient Boosted Trees",
    "Accuracy": round(accuracy,4),
    "F1-Score": round(f1_score,4),
    "ROC-AUC": round(roc_auc,4),
    "Training Time (s)": round(gbt_time,2),
    "Best Hyperparameters":
        f"maxDepth={best_gbt.getMaxDepth()}, "
        f"maxIter={best_gbt.getMaxIter()}, "
        f"stepSize={best_gbt.getStepSize()}"
})

print("="*80)
print("COMMIT 5 COMPLETED")
print("="*80)